# GAN Evaluation — All Runs

Evaluates all trained runs and saves plots to each run's artifact folder.

In [5]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import stats

REPO_ROOT = Path("..")
sys.path.insert(0, str(REPO_ROOT))

from FamaFrenchModel.tcn_marketgan_model import (
    MarketGANConfig,
    prepare_marketgan_data,
    MarketGANSequenceDataset,
    MarketGenerator,
    MarketCritic,
    MarketGANTrainer,
)

ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "marketgan_tcn"
N_SAMPLES     = 50
N_BOOTSTRAP   = 500

In [ ]:
# ── Run configurations ────────────────────────────────────────────────────────
RUN_CONFIGS = [
    {
        "run_name": "marketgan_famafrench_run1",
        "label":    "FamaFrench Run 1 — FF3 Factors + Welch-Goyal Macro (200 epochs)",
        "checkpoint": "latest.pt",   # switch to "fine_tuned.pt" after fine-tuning completes
    },
]

for rc in RUN_CONFIGS:
    cp = ARTIFACT_ROOT / rc["run_name"] / "checkpoints" / rc["checkpoint"]
    print(f"{rc['label']:65s}  checkpoint={'✓' if cp.exists() else '✗ MISSING'}")

FamaFrench Run 1 — FF3 Factors + Welch-Goyal Macro (200 epochs)    checkpoint=✓


In [ ]:
def evaluate_run(rc, artifact_root, n_samples=50, n_bootstrap=500, split="val"):
    """
    split="all"   → sample across full dataset (in-sample, biased)
    split="train" → sample from training windows only
    split="val"   → sample from validation windows only (honest generalization test)
    """
    run_name   = rc["run_name"]
    label      = rc["label"]
    out_dir    = artifact_root / run_name
    checkpoint = artifact_root / run_name / "checkpoints" / rc["checkpoint"]

    if not checkpoint.exists():
        print(f"Skipping {run_name} — {rc['checkpoint']} not found")
        return None

    config = MarketGANConfig(batch_size=128, target_horizon=252 * 4)
    prepared_data = prepare_marketgan_data(config=config, factor_frame=None)
    dataset = MarketGANSequenceDataset(prepared_data, sequence_length=config.total_sequence_length)

    generator = MarketGenerator(prepared_data.dimensions, config).to(config.device)
    ckpt = torch.load(checkpoint, map_location=config.device, weights_only=False)
    generator.load_state_dict(ckpt["generator_state_dict"])
    generator.eval()

    # ── Select window indices based on split ──────────────────────
    total      = len(dataset)
    train_end  = int(total * 0.875)   # first 87.5% = train, last 12.5% = val

    if split == "val":
        idx_range = (train_end, total - 1)
        split_label = f"Val windows ({train_end}–{total-1})"
    elif split == "train":
        idx_range = (0, train_end - 1)
        split_label = f"Train windows (0–{train_end-1})"
    else:
        idx_range = (0, total - 1)
        split_label = f"All windows (0–{total-1})"

    indices = np.linspace(idx_range[0], idx_range[1], n_samples, dtype=int)

    print(f"\n{'='*60}")
    print(f"{label}  (checkpoint epoch: {ckpt['epoch']})")
    print(f"  Split: {split_label}  →  {len(indices)} samples")
    print(f"  Assets: {prepared_data.dimensions.num_assets}  "
          f"Factors: {prepared_data.factor_columns}  "
          f"Covariates: {prepared_data.covariate_columns}")

    # ── Generate samples ──────────────────────────────────────────
    generated_list, real_list = [], []
    with torch.no_grad():
        for idx in indices:
            sample = dataset[int(idx)]
            batch  = {k: v.unsqueeze(0).to(config.device) for k, v in sample.items()}
            out    = generator(
                covariates=batch["covariates"],
                factor_returns=batch["factors"],
                alpha_hat=batch["alpha_hat"],
                beta_hat=batch["beta_hat"],
                sigma_hat=batch["sigma_hat"],
                trim_output=True,
            )
            generated_list.append(out["generated_returns_trimmed"][0].cpu().numpy())
            real_list.append(batch["real_returns"][0, config.warmup_period:, :].cpu().numpy())

    gen_df  = pd.DataFrame(np.concatenate(generated_list, axis=0), columns=prepared_data.asset_columns)
    real_df = pd.DataFrame(np.concatenate(real_list,      axis=0), columns=prepared_data.asset_columns)

    out_dir.mkdir(parents=True, exist_ok=True)
    gen_df.to_csv(out_dir / f"generated_returns_{split}.csv", index=False)

    full_label = f"{label}\n[{split.upper()} SET]"

    # ── Plot 1: Center Mass ────────────────────────────────────────
    s_real = pd.DataFrame({"mean": real_df.mean(), "std": real_df.std(), "skew": real_df.skew(), "kurtosis": real_df.kurtosis()})
    s_gen  = pd.DataFrame({"mean": gen_df.mean(),  "std": gen_df.std(),  "skew": gen_df.skew(),  "kurtosis": gen_df.kurtosis()})

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, m, t in zip(axes.flat,
                        ["mean", "std", "skew", "kurtosis"],
                        ["Mean return", "Volatility (std)", "Skewness", "Excess Kurtosis"]):
        ax.scatter(s_real[m], s_gen[m], alpha=0.7, s=40)
        lo = min(s_real[m].min(), s_gen[m].min())
        hi = max(s_real[m].max(), s_gen[m].max())
        ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="perfect")
        ax.set_xlabel("Real"); ax.set_ylabel("Generated"); ax.set_title(t)
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f"{full_label} — Center Mass", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_dir / f"eval_center_mass_{split}.png", dpi=150)
    plt.show()

    # ── Plot 2: Left Tail Exceedance ───────────────────────────────
    thresholds  = np.linspace(-0.10, -0.005, 60)
    flat_real   = real_df.values.flatten()
    flat_gen    = gen_df.values.flatten()
    real_exc    = np.array([(flat_real < t).mean() for t in thresholds])
    gen_exc     = np.array([(flat_gen  < t).mean() for t in thresholds])
    boot_curves = np.array([
        [(np.random.choice(flat_real, size=len(flat_real), replace=True) < t).mean() for t in thresholds]
        for _ in range(n_bootstrap)
    ])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(thresholds, real_exc, "b-",  linewidth=2, label="Real")
    ax.plot(thresholds, gen_exc,  "r--", linewidth=2, label="Generated")
    ax.fill_between(thresholds,
                    np.percentile(boot_curves, 2.5,  axis=0),
                    np.percentile(boot_curves, 97.5, axis=0),
                    alpha=0.2, color="blue", label="Real 95% CI (bootstrap)")
    ax.set_xlabel("Return threshold"); ax.set_ylabel("Exceedance probability")
    ax.set_title(f"{full_label} — Left Tail Exceedance")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"eval_left_tail_{split}.png", dpi=150)
    plt.show()

    # ── Plot 3: Correlation Matrix ─────────────────────────────────
    corr_real = real_df.corr()
    corr_gen  = gen_df.corr()
    diff      = corr_gen - corr_real
    frob      = np.linalg.norm(diff.values, "fro")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, mat, title in zip(axes,
                               [corr_real, corr_gen, diff],
                               ["Real", "Generated", "Difference (Gen − Real)"]):
        vmin, vmax = (-1, 1) if "Diff" not in title else (-0.5, 0.5)
        im = ax.imshow(mat, vmin=vmin, vmax=vmax, cmap="RdBu_r")
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax)
    plt.suptitle(f"{full_label} — Correlation Matrix  (Frobenius diff: {frob:.3f})", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_dir / f"eval_correlation_{split}.png", dpi=150)
    plt.show()

    # ── Plot 4: Per-Asset Volatility ───────────────────────────────
    vol_real  = real_df.std()
    vol_gen   = gen_df.std()
    boot_vols = pd.DataFrame([
        real_df.iloc[np.random.choice(len(real_df), size=len(real_df), replace=True)].std()
        for _ in range(n_bootstrap)
    ])

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(prepared_data.asset_columns))
    ax.bar(x - 0.2, vol_real, 0.35, label="Real",      color="steelblue", alpha=0.8)
    ax.bar(x + 0.2, vol_gen,  0.35, label="Generated", color="tomato",    alpha=0.8)
    ax.errorbar(x - 0.2, vol_real,
                yerr=[vol_real - boot_vols.quantile(0.025), boot_vols.quantile(0.975) - vol_real],
                fmt="none", color="black", capsize=3, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(prepared_data.asset_columns, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Daily return std")
    ax.set_title(f"{full_label} — Per-Asset Volatility (with 95% CI)")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"eval_volatility_{split}.png", dpi=150)
    plt.show()

    print(f"\n  Frobenius norm (corr diff): {frob:.4f}")
    print(f"  Median vol ratio (gen/real): {(vol_gen / vol_real).median():.4f}")
    print(f"  Saved plots → {out_dir}")
    return {"gen_df": gen_df, "real_df": real_df, "frob": frob, "vol_real": vol_real, "vol_gen": vol_gen}

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────
# Change split= to "all" / "train" / "val" as needed
# "val" is the honest generalization test — model never saw these windows during training

results_val   = {}
results_train = {}

for rc in RUN_CONFIGS:
    print("\n── VAL SET (generalization) ──────────────────────────────")
    results_val[rc["run_name"]]   = evaluate_run(rc, ARTIFACT_ROOT, n_samples=N_SAMPLES, n_bootstrap=N_BOOTSTRAP, split="val")

    print("\n── TRAIN SET (in-sample, reference only) ─────────────────")
    results_train[rc["run_name"]] = evaluate_run(rc, ARTIFACT_ROOT, n_samples=N_SAMPLES, n_bootstrap=N_BOOTSTRAP, split="train")

# ── Summary comparison ────────────────────────────────────────────────────────
print("\n\n── Summary ──────────────────────────────────────────────────")
print(f"{'Run':<45}  {'Split':<8}  {'Frob diff':>10}  {'Vol ratio (med)':>15}")
print("-" * 85)
for rc in RUN_CONFIGS:
    for split, results in [("val", results_val), ("train", results_train)]:
        r = results.get(rc["run_name"])
        if r is None:
            continue
        vol_ratio = (r["vol_gen"] / r["vol_real"]).median()
        print(f"{rc['label']:<45}  {split:<8}  {r['frob']:>10.4f}  {vol_ratio:>15.4f}")

print("\nNote: if val Frob >> train Frob → overfitting. If similar → model generalizes.")

In [ ]:
def plot_rolling_correlation(rc, artifact_root, results_by_split, window=63):
    """
    Rolling correlation analysis — professor's stricter test.
    Computes rolling Frobenius norm of (corr_gen - corr_real) over time.
    Also plots V vs MA rolling pairwise correlation.

    window: rolling window in trading days (63 = 1 quarter, 126 = 6 months)
    """
    run_name  = rc["run_name"]
    label     = rc["label"]
    out_dir   = artifact_root / run_name

    # Use val split results (honest test)
    r = results_by_split.get(run_name)
    if r is None:
        print(f"Skipping {run_name} — run evaluate_run first")
        return

    real_df = r["real_df"].reset_index(drop=True)
    gen_df  = r["gen_df"].reset_index(drop=True)
    T       = len(real_df)

    print(f"  Asset columns: {real_df.columns.tolist()}")

    # ── Rolling Frobenius norm over time ──────────────────────────
    frob_series = []
    timestamps  = []

    for t in range(window, T):
        real_window = real_df.iloc[t - window : t]
        gen_window  = gen_df.iloc[t - window : t]
        corr_r = real_window.corr().values
        corr_g = gen_window.corr().values
        frob   = np.linalg.norm(corr_g - corr_r, "fro")
        frob_series.append(frob)
        timestamps.append(t)

    frob_series = np.array(frob_series)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(timestamps, frob_series, linewidth=0.9, color="steelblue", label="Rolling Frob norm")
    ax.axhline(frob_series.mean(), color="red", linestyle="--", linewidth=1,
               label=f"Mean = {frob_series.mean():.3f}")
    ax.fill_between(timestamps, frob_series, frob_series.mean(),
                    where=frob_series > frob_series.mean(), alpha=0.2, color="red")
    ax.set_xlabel("Timestep (val set)")
    ax.set_ylabel("Frobenius norm")
    ax.set_title(f"{label}\nRolling Correlation Difference (window={window}d) — Real vs Generated [VAL SET]")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"eval_rolling_frob_{window}d.png", dpi=150)
    plt.show()

    # ── V vs MA rolling pairwise correlation ──────────────────────
    # columns are lowercase in the CSV (v, ma)
    a, b = "v", "ma"
    if a not in real_df.columns or b not in real_df.columns:
        print(f"  Warning: {a} or {b} not found in asset columns — skipping pair plot")
    else:
        roll_real = real_df[[a, b]].rolling(window).corr().unstack()[a][b].dropna()
        roll_gen  = gen_df[[a, b]].rolling(window).corr().unstack()[a][b].dropna()

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(roll_real.values, "b-",  linewidth=1.0, label="Real",      alpha=0.85)
        ax.plot(roll_gen.values,  "r--", linewidth=1.0, label="Generated", alpha=0.85)
        ax.set_title(f"{label}\nRolling Correlation: Visa (v) vs Mastercard (ma) (window={window}d) [VAL SET]")
        ax.set_xlabel("Timestep (val set)")
        ax.set_ylabel("Correlation")
        ax.set_ylim(-1, 1)
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / f"eval_rolling_pair_V_MA_{window}d.png", dpi=150)
        plt.show()

        print(f"  V vs MA — real mean corr: {roll_real.mean():.4f}  gen mean corr: {roll_gen.mean():.4f}")

    print(f"  Rolling Frob — mean: {frob_series.mean():.4f}  "
          f"std: {frob_series.std():.4f}  "
          f"max: {frob_series.max():.4f}")
    print(f"  Saved → {out_dir}")


# ── Run rolling correlation on val results ────────────────────────────────────
for rc in RUN_CONFIGS:
    plot_rolling_correlation(rc, ARTIFACT_ROOT, results_val, window=63)